# DEH 30-Day PySpark Challenge
## Day 24 — Practice Set 3: Window Functions

**Instructions:**
1. `File → Save a copy in Drive` first
2. SQL then PySpark for each problem
3. Reference solutions follow each problem

---

In [ ]:
!pip install pyspark --quiet

In [ ]:
import os
os.environ['AWS_ACCESS_KEY_ID']     = 'your-access-key-here'
os.environ['AWS_SECRET_ACCESS_KEY'] = 'your-secret-key-here'
os.environ['AWS_DEFAULT_REGION']    = 'us-east-1'
BUCKET = 'deh-pyspark-challenge-your-account-id'
print('Credentials set.')

In [ ]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DoubleType, DateType

spark = (
    SparkSession.builder.appName("DEH-PySpark-Challenge")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.4.0,com.amazonaws:aws-java-sdk-bundle:1.12.367")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.aws.credentials.provider", "com.amazonaws.auth.EnvironmentVariableCredentialsProvider")
    .getOrCreate()
)
print(f"Spark version: {spark.version}")

In [ ]:
orders_schema = StructType([
    StructField("order_id", StringType(), True), StructField("customer_id", StringType(), True),
    StructField("product_id", StringType(), True), StructField("order_date", DateType(), True),
    StructField("quantity", IntegerType(), True), StructField("unit_price", DoubleType(), True),
    StructField("discount_pct", IntegerType(), True), StructField("status", StringType(), True),
    StructField("payment_method", StringType(), True), StructField("region", StringType(), True)
])
orders_df = spark.read.option("header","true").schema(orders_schema).csv(f"s3a://{BUCKET}/data/orders.csv")
orders_df.createOrReplaceTempView("orders")
print(f"Orders: {orders_df.count()}")

---
## Problem 1 (Easy) — Top order per customer

For each customer, find their highest-value order. Return `customer_id`, `order_id`, `unit_price`.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 1

In [ ]:
spark.sql("""
    SELECT customer_id, order_id, unit_price FROM (
        SELECT customer_id, order_id, unit_price,
               ROW_NUMBER() OVER (PARTITION BY customer_id ORDER BY unit_price DESC) AS rn
        FROM orders
    ) WHERE rn = 1
""").show(10)

In [ ]:
w = Window.partitionBy("customer_id").orderBy(F.col("unit_price").desc())
orders_df.withColumn("rn", F.row_number().over(w)) \
    .filter(F.col("rn") == 1) \
    .select("customer_id", "order_id", "unit_price") \
    .show(10)

---
## Problem 2 (Easy) — Second highest order value per region

Find the order with the 2nd highest `unit_price` per region. Consider ties.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 2

Use `dense_rank()` — if two orders tie for highest, both get rank 1, and the next distinct value gets rank 2 (the true "second highest value"). `rank()` would skip to 3 instead.

In [ ]:
spark.sql("""
    SELECT region, order_id, unit_price FROM (
        SELECT region, order_id, unit_price,
               DENSE_RANK() OVER (PARTITION BY region ORDER BY unit_price DESC) AS rnk
        FROM orders
    ) WHERE rnk = 2
""").show()

In [ ]:
w = Window.partitionBy("region").orderBy(F.col("unit_price").desc())
orders_df.withColumn("rnk", F.dense_rank().over(w)) \
    .filter(F.col("rnk") == 2) \
    .select("region", "order_id", "unit_price") \
    .show()

---
## Problem 3 (Medium) — Running total of revenue by date

No partition — running total across the entire dataset ordered by `order_date`.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 3

In [ ]:
spark.sql("""
    SELECT order_date, unit_price * quantity AS revenue,
           SUM(unit_price * quantity) OVER (
               ORDER BY order_date
               ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
           ) AS running_total
    FROM orders
    ORDER BY order_date
""").show(10)

In [ ]:
w = Window.orderBy("order_date").rowsBetween(Window.unboundedPreceding, Window.currentRow)
orders_df.withColumn("revenue", F.col("unit_price") * F.col("quantity")) \
    .withColumn("running_total", F.round(F.sum("revenue").over(w), 2)) \
    .select("order_date", "revenue", "running_total") \
    .orderBy("order_date") \
    .show(10)

---
## Problem 4 (Medium) — Month-over-month change per region

Monthly revenue per region, with `lag()` showing previous month and the delta.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 4

In [ ]:
spark.sql("""
    WITH monthly AS (
        SELECT region, DATE_FORMAT(order_date, 'yyyy-MM') AS year_month,
               SUM(unit_price * quantity) AS revenue
        FROM orders
        GROUP BY region, DATE_FORMAT(order_date, 'yyyy-MM')
    )
    SELECT region, year_month, ROUND(revenue, 2) AS revenue,
           ROUND(LAG(revenue) OVER (PARTITION BY region ORDER BY year_month), 2) AS prev_month_revenue,
           ROUND(revenue - LAG(revenue) OVER (PARTITION BY region ORDER BY year_month), 2) AS change
    FROM monthly
    ORDER BY region, year_month
""").show(20)

In [ ]:
monthly = orders_df.withColumn("year_month", F.date_format("order_date", "yyyy-MM")) \
    .groupBy("region", "year_month").agg(
        F.round(F.sum(F.col("unit_price") * F.col("quantity")), 2).alias("revenue")
    )

w = Window.partitionBy("region").orderBy("year_month")
monthly.withColumn("prev_month_revenue", F.lag("revenue").over(w)) \
    .withColumn("change", F.round(F.col("revenue") - F.col("prev_month_revenue"), 2)) \
    .orderBy("region", "year_month") \
    .show(20)

---
## Problem 5 (Hard) — Percentage of region's total per order

For every order, what % of its region's total revenue does it represent?

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 5

A window with `partitionBy()` but no `orderBy()` aggregates over the WHOLE partition for every row — this gives the region total alongside each individual order.

In [ ]:
spark.sql("""
    SELECT order_id, region, unit_price * quantity AS order_revenue,
           SUM(unit_price * quantity) OVER (PARTITION BY region) AS region_total,
           ROUND(100.0 * (unit_price * quantity) / SUM(unit_price * quantity) OVER (PARTITION BY region), 2) AS pct_of_region
    FROM orders
    ORDER BY region, pct_of_region DESC
""").show(10)

In [ ]:
w = Window.partitionBy("region")  # no orderBy — aggregates over the whole partition

orders_df.withColumn("order_revenue", F.col("unit_price") * F.col("quantity")) \
    .withColumn("region_total", F.sum("order_revenue").over(w)) \
    .withColumn("pct_of_region", F.round(100.0 * F.col("order_revenue") / F.col("region_total"), 2)) \
    .select("order_id", "region", "order_revenue", "region_total", "pct_of_region") \
    .orderBy("region", F.col("pct_of_region").desc()) \
    .show(10)

---
## Problem 6 (Hard) — Detect consecutive order value increases

Flag orders where price increased vs the previous order per customer. Count streaks of length >= 2.

In [ ]:
# Your SQL attempt


In [ ]:
# Your PySpark attempt


### Reference Solution — Problem 6

Approach: use `lag()` to flag increases, then use the classic "groups" trick — a running count of NON-increase flags creates a group id; rows within the same increase-streak share a group id, which lets you count streak lengths with a second groupBy.

In [ ]:
w_order = Window.partitionBy("customer_id").orderBy("order_date")

# Step 1 — flag whether this order increased vs previous
flagged = orders_df.withColumn("prev_price", F.lag("unit_price").over(w_order)) \
    .withColumn("is_increase", F.when(F.col("unit_price") > F.col("prev_price"), 1).otherwise(0))

# Step 2 — running count of non-increases creates a streak group id
w_running = Window.partitionBy("customer_id").orderBy("order_date") \
    .rowsBetween(Window.unboundedPreceding, Window.currentRow)

grouped = flagged.withColumn(
    "streak_group",
    F.sum(F.when(F.col("is_increase") == 0, 1).otherwise(0)).over(w_running)
)

# Step 3 — count streak lengths per customer per streak_group, keep streaks >= 2
streaks = grouped.filter(F.col("is_increase") == 1) \
    .groupBy("customer_id", "streak_group").agg(F.count("*").alias("streak_length")) \
    .filter(F.col("streak_length") >= 1)  # length here counts increase-rows in the streak

streaks.groupBy("customer_id").agg(F.count("*").alias("num_increase_streaks")) \
    .orderBy(F.col("num_increase_streaks").desc()) \
    .show(10)

---
*DEH 30-Day PySpark Challenge · Data Engineering Hub · RADE Program*